# AV-Forensics — Colab Training Notebook

**Conference Paper 2**: Audio-Visual Lip-Sync Consistency Transformer for Deepfake Detection and Modality-Level Forgery Classification.

Dataset: FakeAVCeleb (Kaggle: `shreyaty08/fakeavceleb`)  
Audio Encoder: `utter-project/mHuBERT-147`  
Runtime: T4 GPU (free tier)

---

### Before you start
1. Enable GPU: Runtime → Change runtime type → T4 GPU
2. Mount Google Drive (cell below) so checkpoints survive restarts
3. Set your Kaggle credentials in the env vars cell


## Cell 1 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
SAVE_DIR = '/content/drive/MyDrive/avforensics_checkpoints'
os.makedirs(SAVE_DIR, exist_ok=True)
print('Drive mounted. Checkpoints →', SAVE_DIR)

## Cell 2 — Install Libraries

In [ ]:
!pip install -q kagglehub
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install -q transformers==4.40.2
!pip install -q opencv-python-headless==4.9.0.80
!pip install -q mediapipe==0.10.11
!pip install -q librosa==0.10.2 soundfile==0.12.1
!pip install -q scikit-learn==1.4.2
!pip install -q matplotlib==3.8.4 seaborn==0.13.2
!pip install -q pandas==2.2.2 tqdm==4.66.4
!pip install -q torchinfo==1.8.0
!pip install -q reportlab==4.2.0
!apt-get install -qq ffmpeg
print('All libraries installed.')

## Cell 3 — Imports and Configuration

In [ ]:
import os, sys, glob, json, random, logging, time, shutil
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchinfo import summary
from tqdm import tqdm

logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

# ── Add ml/ to sys.path so all modules are importable ──
ML_DIR = '/content/avforensics/ml'   # adjust if you upload the repo elsewhere
if ML_DIR not in sys.path:
    sys.path.insert(0, ML_DIR)

# ── Central config ──
# All constants are in config.py.  Only DATA_ROOT changes per environment.
from config import (
    AUDIO_MODEL_NAME, CLASSES, CLASS_TO_ID, BINARY_LABEL, AUDIO_FAKE, VIDEO_FAKE,
    SR, N_FRAMES, IMG_SIZE, AUDIO_LEN, WINDOW_SEC, STRIDE_SEC,
    EMBED_DIM, LAMBDA_SYNC, AUX_WEIGHT,
    EPOCHS, BATCH_SIZE, GRAD_ACCUM, PATIENCE, VAL_FRAC,
    FAKE_THRESHOLD, TOP_EVIDENCE_N,
)

DATA_ROOT = '/content/FakeAVCeleb_Dataset'
CACHE_DIR = '/content/drive/MyDrive/avforensics_cache'
os.makedirs(CACHE_DIR, exist_ok=True)

print('Config loaded.')
print(f'Audio model : {AUDIO_MODEL_NAME}')
print(f'N_FRAMES    : {N_FRAMES}')
print(f'IMG_SIZE    : {IMG_SIZE}')
print(f'SR          : {SR}')
print(f'AUDIO_LEN   : {AUDIO_LEN} samples ({AUDIO_LEN/SR:.1f} s)')

## Cell 4 — Download and Organise Dataset from Kaggle

In [ ]:
# Set Kaggle credentials (or upload ~/.kaggle/kaggle.json)
import os
os.environ['KAGGLE_USERNAME'] = 'YOUR_KAGGLE_USERNAME'  # <-- fill in
os.environ['KAGGLE_KEY']      = 'YOUR_KAGGLE_API_KEY'   # <-- fill in

import kagglehub
raw_path = kagglehub.dataset_download('shreyaty08/fakeavceleb')
print('Downloaded to:', raw_path)

# ── Map Kaggle folder names → short names ──────────────────────────────────
# Kaggle layout may vary. Adjust the mapping below to match what you see.
KAGGLE_MAP = {
    'RealVideo-RealAudio': 'RVRA',
    'RealVideo-FakeAudio': 'RVFA',
    'FakeVideo-RealAudio': 'FVRA',
    'FakeVideo-FakeAudio': 'FVFA',
}

os.makedirs(DATA_ROOT, exist_ok=True)

for kaggle_name, short_name in KAGGLE_MAP.items():
    src = os.path.join(raw_path, kaggle_name)
    if not os.path.isdir(src):
        # Try recursive search
        matches = glob.glob(os.path.join(raw_path, '**', kaggle_name), recursive=True)
        src = matches[0] if matches else None
    if src is None:
        print(f'WARNING: could not find {kaggle_name}')
        continue

    videos = sorted(glob.glob(os.path.join(src, '**', '*.mp4'), recursive=True))
    random.seed(42)
    random.shuffle(videos)
    n_train = int(len(videos) * 0.8)
    splits = {'train': videos[:n_train], 'test': videos[n_train:]}

    for split, paths in splits.items():
        dst_dir = os.path.join(DATA_ROOT, short_name, split)
        os.makedirs(dst_dir, exist_ok=True)
        for p in paths:
            dst = os.path.join(dst_dir, os.path.basename(p))
            if not os.path.exists(dst):
                shutil.copy2(p, dst)
    print(f'{kaggle_name} → {short_name}: {len(splits["train"])} train / {len(splits["test"])} test')

## Cell 5 — Dataset Counts and Validation

In [ ]:
rows = []
for c in CLASSES:
    for s in ['train', 'test']:
        n = len(glob.glob(os.path.join(DATA_ROOT, c, s, '*.mp4')))
        rows.append({'class': c, 'split': s, 'videos': n})

df = pd.DataFrame(rows)
pivot = df.pivot(index='class', columns='split', values='videos')
print('Video counts per class and split:')
print(pivot)

# Bar chart
pivot.plot(kind='bar', figsize=(8, 4), rot=0, title='FakeAVCeleb — Video Counts')
plt.ylabel('Number of Videos')
plt.tight_layout()
plt.savefig('/content/dataset_counts.png', dpi=120)
plt.show()

# Quick integrity check: does each file open and has audio?
print('\nRunning quick integrity check (first 20 files)...')
from preprocessing import load_audio_16k
all_mp4 = glob.glob(os.path.join(DATA_ROOT, '*', '*', '*.mp4'))
bad = []
for p in all_mp4[:20]:
    cap = cv2.VideoCapture(p)
    ok = cap.isOpened()
    cap.release()
    wav = load_audio_16k(p, AUDIO_LEN, SR)
    if not ok or wav is None:
        bad.append(p)

print(f'Bad files (no face/audio): {len(bad)}')
for b in bad:
    print(' ', b)

## Cell 6 — Visualise Samples

In [ ]:
from preprocessing import load_lip_frames, load_audio_16k
import librosa

fig, axes = plt.subplots(len(CLASSES), 4, figsize=(16, len(CLASSES) * 3))

for row, cls in enumerate(CLASSES):
    videos = glob.glob(os.path.join(DATA_ROOT, cls, 'train', '*.mp4'))
    if not videos:
        continue
    vp = videos[0]

    # Raw frame
    cap = cv2.VideoCapture(vp)
    ok, frame = cap.read(); cap.release()
    if ok:
        axes[row][0].imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    axes[row][0].set_title(f'{cls} — Frame')
    axes[row][0].axis('off')

    # Mouth crop
    frames_arr, raws = load_lip_frames(vp, N_FRAMES, IMG_SIZE)
    if frames_arr is not None:
        axes[row][1].imshow(frames_arr[N_FRAMES // 2, 0], cmap='gray')
    axes[row][1].set_title(f'{cls} — Mouth Crop')
    axes[row][1].axis('off')

    # All mouth crops strip
    if frames_arr is not None:
        strip = np.concatenate([
            frames_arr[i, 0] for i in range(0, N_FRAMES, N_FRAMES // 8)
        ], axis=1)
        axes[row][2].imshow(strip, cmap='gray', aspect='auto')
    axes[row][2].set_title(f'{cls} — Frame Strip')
    axes[row][2].axis('off')

    # Audio waveform
    wav = load_audio_16k(vp, AUDIO_LEN, SR)
    if wav is not None:
        axes[row][3].plot(wav[:8000], linewidth=0.5)
        axes[row][3].set_title(f'{cls} — Audio (first 0.5s)')
        axes[row][3].set_xlabel('Sample')

plt.tight_layout()
plt.savefig('/content/samples.png', dpi=120)
plt.show()

## Cell 7 — Preprocessing: Cache Frames and Audio to .npy

In [ ]:
# This cell runs once and caches preprocessed arrays to Drive.
# Re-running the notebook skips cached files.
from dataset import scan_split
from preprocessing import preprocess_video

all_items = scan_split(DATA_ROOT, 'train') + scan_split(DATA_ROOT, 'test')
print(f'Total videos: {len(all_items)}')

skipped = []
for path, cls_name in tqdm(all_items, desc='Preprocessing'):
    stem   = Path(path).stem
    parent = Path(path).parent.name
    cls    = Path(path).parent.parent.name
    d = Path(CACHE_DIR) / cls / parent
    d.mkdir(parents=True, exist_ok=True)
    cp_f = d / f'{stem}_frames.npy'
    cp_a = d / f'{stem}_audio.npy'

    if cp_f.exists() and cp_a.exists():
        continue   # already cached

    result = preprocess_video(str(path), N_FRAMES, IMG_SIZE, AUDIO_LEN, SR)
    if result['ok']:
        np.save(str(cp_f), result['frames'])
        np.save(str(cp_a), result['audio'])
    else:
        skipped.append((path, result['error']))

print(f'Done. Skipped {len(skipped)} files:')
for p, err in skipped[:10]:
    print(f'  {p}: {err}')

## Cell 8 — Build Datasets and DataLoaders

In [ ]:
from dataset import scan_split, identity_grouped_split, compute_class_weights, AVDataset

train_items_raw = scan_split(DATA_ROOT, 'train')
test_items      = scan_split(DATA_ROOT, 'test')

train_items, val_items = identity_grouped_split(train_items_raw, val_frac=VAL_FRAC)

print(f'Train: {len(train_items)} | Val: {len(val_items)} | Test: {len(test_items)}')

class_weights = compute_class_weights(train_items)
print('Class weights (4-class):', class_weights.numpy())

train_ds = AVDataset(train_items, cache_dir=CACHE_DIR)
val_ds   = AVDataset(val_items,   cache_dir=CACHE_DIR)
test_ds  = AVDataset(test_items,  cache_dir=CACHE_DIR)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print('DataLoaders ready.')

## Cell 9 — Build Model and Print Summary

In [ ]:
from model import AVDeepfakeModel

model = AVDeepfakeModel(
    embed_dim=EMBED_DIM,
    audio_model=AUDIO_MODEL_NAME,
    freeze_cnn=True,
    freeze_layers=6,
).to(device)

print('Model built.')

# Count trainable parameters
n_params       = sum(p.numel() for p in model.parameters())
n_trainable    = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total params     : {n_params:,}')
print(f'Trainable params : {n_trainable:,}')

# Verify output shapes with a dummy batch
with torch.no_grad():
    dummy_frames = torch.randn(2, N_FRAMES, 1, IMG_SIZE, IMG_SIZE).to(device)
    dummy_audio  = torch.randn(2, AUDIO_LEN).to(device)
    dummy_out    = model(dummy_frames, dummy_audio)

print('Shape assertions:')
assert dummy_out['logits_bin'].shape   == (2, 2)
assert dummy_out['logits_4'].shape     == (2, 4)
assert dummy_out['logits_afake'].shape == (2, 2)
assert dummy_out['logits_vfake'].shape == (2, 2)
assert dummy_out['vis_emb'].shape[0]   == 2
assert dummy_out['aud_emb'].shape[0]   == 2
print('  ✓ logits_bin   :', dummy_out['logits_bin'].shape)
print('  ✓ logits_4     :', dummy_out['logits_4'].shape)
print('  ✓ logits_afake :', dummy_out['logits_afake'].shape)
print('  ✓ logits_vfake :', dummy_out['logits_vfake'].shape)
print('  ✓ vis_emb      :', dummy_out['vis_emb'].shape)
print('  ✓ aud_emb      :', dummy_out['aud_emb'].shape)

## Cell 10 — Train

In [ ]:
from train import train

history = train(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    device=device,
    class_weights_4=class_weights,
    save_dir=SAVE_DIR,
    use_amp=True,
)

print('Training complete.')

## Cell 11 — Full Evaluation on Test Set

In [ ]:
from evaluate import full_evaluation

# Load best weights
best_ckpt = os.path.join(SAVE_DIR, 'best_model.pth')
model.load_state_dict(torch.load(best_ckpt, map_location=device))
model.eval()
print(f'Loaded best weights from {best_ckpt}')

PLOT_DIR = '/content/drive/MyDrive/avforensics_plots'
os.makedirs(PLOT_DIR, exist_ok=True)

results = full_evaluation(
    model=model,
    loader=test_loader,
    device=device,
    history=history,
    plot_dir=PLOT_DIR,
)

## Cell 12 — Export model.pth + config.json

In [ ]:
import json, torch

# Determine best threshold from validation set using Youden's J
from sklearn.metrics import roc_curve
from evaluate import collect_predictions

val_preds = collect_predictions(model, val_loader, device)
fpr, tpr, thresholds = roc_curve(val_preds['bin_true'], val_preds['bin_prob'])
j_idx     = np.argmax(tpr - fpr)
best_thresh = float(thresholds[j_idx])
print(f'Best binary threshold (Youden J): {best_thresh:.4f}')

config = {
    'classes':          CLASSES,
    'binary_label':     BINARY_LABEL,
    'audio_fake':       AUDIO_FAKE,
    'video_fake':       VIDEO_FAKE,
    'sr':               SR,
    'n_frames':         N_FRAMES,
    'img_size':         IMG_SIZE,
    'audio_len':        AUDIO_LEN,
    'window_sec':       WINDOW_SEC,
    'stride_sec':       STRIDE_SEC,
    'embed_dim':        EMBED_DIM,
    'audio_model_name': AUDIO_MODEL_NAME,
    'fake_threshold':   best_thresh,
    'model_version':    '1.0.0',
}

MODEL_OUT  = os.path.join(SAVE_DIR, 'model.pth')
CONFIG_OUT = os.path.join(SAVE_DIR, 'config.json')

torch.save(model.state_dict(), MODEL_OUT)
with open(CONFIG_OUT, 'w') as f:
    json.dump(config, f, indent=2)

print(f'Saved model  → {MODEL_OUT}')
print(f'Saved config → {CONFIG_OUT}')
print(json.dumps(config, indent=2))

## Cell 13 — Test on a Few Held-Out Examples

In [ ]:
from evaluate import collect_predictions
from config import MODALITY_LABELS

# Show 8 examples from test set
model.eval()
n_show = 8
print(f'{'File':<40} {'True':<8} {'Pred':<8} {'Conf':>6}')
print('-' * 65)

shown = 0
with torch.no_grad():
    for path, cls_name in test_items:
        from preprocessing import preprocess_video
        result = preprocess_video(str(path), N_FRAMES, IMG_SIZE, AUDIO_LEN, SR)
        if not result['ok']:
            continue
        frames = torch.from_numpy(result['frames']).unsqueeze(0).to(device)
        audio  = torch.from_numpy(result['audio']).unsqueeze(0).to(device)
        out    = model(frames, audio)

        prob_bin = torch.softmax(out['logits_bin'], 1)[0, 1].item()
        pred_bin = 'FAKE' if prob_bin > FAKE_THRESHOLD else 'REAL'
        true_bin = 'FAKE' if BINARY_LABEL[cls_name] == 1 else 'REAL'

        fname = Path(path).name[:38]
        print(f'{fname:<40} {true_bin:<8} {pred_bin:<8} {prob_bin:>6.3f}')
        shown += 1
        if shown >= n_show:
            break

## Cell 14 — Manual Upload Test (Single Video)

In [ ]:
from google.colab import files
from preprocessing import preprocess_video
from config import MODALITY_LABELS, CLASSES, FAKE_THRESHOLD

uploaded = files.upload()
video_path = list(uploaded.keys())[0]

result = preprocess_video(video_path, N_FRAMES, IMG_SIZE, AUDIO_LEN, SR)
if not result['ok']:
    print('ERROR:', result['error'])
else:
    model.eval()
    with torch.no_grad():
        frames = torch.from_numpy(result['frames']).unsqueeze(0).to(device)
        audio  = torch.from_numpy(result['audio']).unsqueeze(0).to(device)
        out    = model(frames, audio)

    import torch.nn.functional as F

    prob_bin  = F.softmax(out['logits_bin'], 1)[0]
    prob_4    = F.softmax(out['logits_4'],   1)[0]
    pred_cls  = CLASSES[prob_4.argmax().item()]

    # Per-step sync score
    v_n = F.normalize(out['vis_emb'][0], dim=-1)
    a_n = F.normalize(out['aud_emb'][0], dim=-1)
    sync_scores = (v_n * a_n).sum(-1).cpu().numpy()

    pred_verdict = 'FAKE' if prob_bin[1].item() > FAKE_THRESHOLD else 'REAL'

    output = {
        'verdict':    pred_verdict,
        'confidence': round(prob_bin[1].item() if pred_verdict == 'FAKE' else prob_bin[0].item(), 4),
        'modality':   pred_cls,
        'modality_label': MODALITY_LABELS[pred_cls],
        'class_probs': {c: round(p, 4) for c, p in zip(CLASSES, prob_4.tolist())},
        'sync_score_mean': round(float(sync_scores.mean()), 4),
        'sync_timeline': sync_scores.tolist(),
    }
    import json; print(json.dumps(output, indent=2))

    # Plot sync timeline
    plt.figure(figsize=(10, 3))
    plt.plot(sync_scores, marker='o', markersize=3)
    plt.axhline(0.5, color='red', linestyle='--', label='Threshold')
    plt.title(f'Lip-Audio Sync Timeline — {pred_verdict} ({pred_cls})')
    plt.xlabel('Frame')
    plt.ylabel('Cosine Similarity')
    plt.legend()
    plt.tight_layout()
    plt.show()